# 07 · Tarea 4 — Incertidumbre: clase + varianza

**Objetivo.** A partir de la **topologia con dropout que fija el tuner** (notebook 06;
palanca compartida **D-3.2 <-> D-4.1**), producir sobre `test` una prediccion que devuelva
**clase + varianza** mediante **MC-Dropout** (D-4.1): se hacen **T pasadas** dejando el
dropout **activo en inferencia** (`model(X_test, training=True)`); la **media** de las `T`
probabilidades fija la **clase** (umbral `tau`, **D-4.4 Abierta**) y la **`Var[p]`** entre
pasadas mide la **incertidumbre epistemica**. El numero de pasadas **T** se decide en
**D-4.2 (Abierta, sug. 50-100)** comprobando que la varianza se estabiliza. Con eso se
grafica la **distribucion de la varianza "buen pagador" vs "mal pagador"** (entregable
**E3**) y se **cruza la varianza con `N_EXT_MISSING` / flags `*_missing`** (**D-4.3**) para
responder la pregunta del taller: *¿hay mas duda donde faltan las fuentes externas?*
Como **entrega base fiel al profe** se anade el **segundo modelo del error** (`|p-y|`),
que es lo minimo que pide entregar ([T4] §3.2, `docs/teoria/04-incertidumbre.md`).

## Decisiones a tomar antes de empezar

> Fichas de `docs/DECISIONES.md` para esta tarea. **Estado real** copiado tal cual.
> Las decisiones en **Propuesta**/**Abierta** se **validan con el grupo ANTES de
> implementar**: este notebook asume la *Propuesta* por defecto, pero es revisable.

| Decision | Opciones | Estado |
|---|---|---|
| **D-4.1** · Metodo para la varianza | (a) 2o modelo del error / (b) **MC-Dropout** / (c) deep ensemble | Propuesta |
| **D-4.2** · No de pasadas/miembros T | 15 / 100 / **sug. 50-100** comprobando estabilidad | **Abierta** |
| **D-4.3** · Medir calidad de `EXT_SOURCE` | **`N_EXT_MISSING` (0-3) + flags `*_missing`** / flag binario / magnitud imputada | Propuesta |
| **D-4.4** · Umbral tau de clasificacion | 0,5 fijo / **ajustado por coste FN** / por percentil (desbalance 11,4:1 => tau<0,5) | **Abierta** |
| **D-4.5** · Descomposicion aleatoria/epistemica + calibracion | hacerlo / **dejar como extension** | Propuesta (extension) |

> **Nota.** **D-4.1 (MC-Dropout)** reutiliza el **dropout que fija el tuner** (06): es el
> cruce **D-3.2 <-> D-4.1**, no se introduce un dropout nuevo desacoplado. Las decisiones en
> **Propuesta**/**Abierta** se **validan con el grupo antes de implementar**.

In [1]:
# === Setup comun (notebooks de modelado 03-07) ===
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # backend unico para todo el grupo

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibilidad
RNG = 42
np.random.seed(RNG)
import random; random.seed(RNG)
try:
    import keras
    keras.utils.set_random_seed(RNG)
except Exception:
    pass

# Estilo heredado del EDA / preprocesado
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
COLOR_PAGA   = "#2c7fb8"   # TARGET=0  (paga)
COLOR_IMPAGA = "#d7301f"   # TARGET=1  (impaga)
COLOR_ACENTO = "#41ab5d"   # neutro

# Rutas estandar
PROC_DIR = Path("../data/processed")
FIG_DIR  = Path("../results/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR  = Path("../results/tables");  TAB_DIR.mkdir(parents=True, exist_ok=True)

# --- Especifico Tarea 4 ---
import keras
import sys; sys.path.append("..")
try:
    from src import uncertainty
except Exception:
    uncertainty = None   # tolerante: el stub existe, pero por robustez no se exige
print("keras:", keras.__version__, "| uncertainty:", "OK" if uncertainty else "no disponible")

keras: 3.14.0 | uncertainty: OK


In [2]:
import json
from pathlib import Path
import pandas as pd

# --- Rutas y metadatos (fuente de verdad: metadata.json del preprocesado) ---
PROC_DIR   = Path("../data/processed")                       # relativo a notebooks/
META       = json.loads((PROC_DIR / "metadata.json").read_text(encoding="utf-8"))
FEATURES_X = META["columns"]["features_X"]   # 13 features, en orden
SENSIBLE   = META["columns"]["sensible"]     # "CODE_GENDER"  (s)
TARGET     = META["columns"]["target"]       # "TARGET"       (y)

def cargar_split(nombre):
    """Devuelve (X, y, s) para 'train' | 'val' | 'test'.
    X = DataFrame solo con las 13 features (SIN genero).
    y = Series TARGET (1=impaga, 0=paga).  s = Series CODE_GENDER (M=1/F=0).
    """
    df = pd.read_parquet(PROC_DIR / f"{nombre}.parquet")
    X = df[FEATURES_X]          # input del modelo: el genero NUNCA entra aqui
    y = df[TARGET]
    s = df[SENSIBLE]
    assert SENSIBLE not in X.columns, "FUGA: el genero esta dentro de X"
    return X, y, s

# Materializar los tres cortes
X_train, y_train, s_train = cargar_split("train")
X_val,   y_val,   s_val   = cargar_split("val")
X_test,  y_test,  s_test  = cargar_split("test")

# Resumen de control
print(f"{'split':<7}{'X (filas, cols)':>20}{'y':>12}{'s':>12}{'tasa_impago':>14}")
for n, (X, y, s) in {"train": (X_train, y_train, s_train),
                     "val":   (X_val,   y_val,   s_val),
                     "test":  (X_test,  y_test,  s_test)}.items():
    print(f"{n:<7}{str(tuple(X.shape)):>20}{str(tuple(y.shape)):>12}"
          f"{str(tuple(s.shape)):>12}{y.mean():>14.4%}")

# --- DESTACADO Tarea 4 (D-4.3): X_test YA trae la materia prima para cruzar varianza ---
# N_EXT_MISSING (0-3) + un flag *_missing por fuente externa estan dentro de X_test.
print("\n[D-4.3] Materia prima del cruce varianza <-> EXT_SOURCE (ya en X_test):")
print(X_test[["N_EXT_MISSING", "EXT_SOURCE_1_missing",
              "EXT_SOURCE_2_missing", "EXT_SOURCE_3_missing"]].describe())

split       X (filas, cols)           y           s   tasa_impago
train          (215254, 13)   (215254,)   (215254,)       8.0728%
val             (46126, 13)    (46126,)    (46126,)       8.0735%
test            (46127, 13)    (46127,)    (46127,)       8.0734%

[D-4.3] Materia prima del cruce varianza <-> EXT_SOURCE (ya en X_test):
       N_EXT_MISSING  EXT_SOURCE_1_missing  EXT_SOURCE_2_missing  \
count   46127.000000          46127.000000          46127.000000   
mean        0.765539              0.563249              0.002276   
std         0.651217              0.495989              0.047657   
min         0.000000              0.000000              0.000000   
25%         0.000000              0.000000              0.000000   
50%         1.000000              1.000000              0.000000   
75%         1.000000              1.000000              0.000000   
max         3.000000              1.000000              1.000000   

       EXT_SOURCE_3_missing  
count          46127

## Cargar / heredar el modelo con dropout del tuner (06)

Se **hereda la topologia + dropout** que **fija el notebook 06** (Keras Tuner): es el cruce
**D-3.2 <-> D-4.1**, el mismo dropout que MC-Dropout reutiliza en inferencia; si 06 aun no ha
exportado su artefacto, se documenta un *fallback* (MLP con `Dropout` local) para no bloquear.

In [3]:
# TODO: heredar topologia+dropout que fija el NB 06 (cruce D-3.2 <-> D-4.1);
#       fallback documentado: MLP con Dropout local si 06 aun no exporto artefacto

## MC-Dropout: T pasadas -> `Var[p]`

Se hacen **T pasadas** con `model(X_test, training=True)` (dropout activo en inferencia,
**D-4.1**): la **media** fija la clase (umbral `tau`) y la **varianza** entre pasadas es la
incertidumbre epistemica; **T** se decide en **D-4.2 (Abierta, sug. 50-100)**.

In [4]:
# TODO: en src/uncertainty.py -- mc_dropout_predict(model, X, T, tau):
#       T veces model(X_test, training=True); media -> clase (tau), varianza -> incertidumbre; T por D-4.2

## Segundo modelo del error (entrega base)

Entrega **base fiel al profe** ([T4] §3.2): se crea la variable `error = |p - y|` en
train/val y se entrena un **segundo modelo** que la predice a partir de las mismas `X`.

In [5]:
# TODO: variable error=|p-y| en train/val + 2o modelo que la predice ([T4] §3.2); en src/uncertainty.py

## Distribucion de varianza: buen vs mal pagador (E3)

Entregable **E3**: histogramas/KDE de `Var[p]` por clase (**buen** vs **mal pagador**),
usando la paleta semantica (`COLOR_PAGA` vs `COLOR_IMPAGA`).

In [6]:
# TODO: histogramas/KDE de Var[p] por clase (COLOR_PAGA vs COLOR_IMPAGA)
#       -> results/figures/07_incert__varianza_buen_vs_mal_pagador.png (E3)

## Cruce varianza <-> `EXT_SOURCE` / `N_EXT_MISSING` (D-4.3)

Se cruza `Var[p]` con la calidad de fuentes externas (**D-4.3**): ¿hay mas duda donde
faltan fuentes? Boxplot de `Var[p]` por `N_EXT_MISSING` (0-3) + correlacion.

In [7]:
# TODO: boxplot Var[p] por N_EXT_MISSING (0-3) + correlacion
#       -> results/figures/07_incert__varianza_vs_n_ext_missing.png

## (Extension) Descomposicion aleatoria/epistemica + umbral tau (D-4.5, D-4.4)

Como **extension opcional**: fijar `tau` como decision de politica del grupo (**D-4.4
Abierta**, con desbalance 11,4:1 => `tau<0,5`) y separar incertidumbre aleatoria/epistemica
+ calibracion (**D-4.5**, mejora no obligatoria).

In [8]:
# TODO opcional: tau decision de politica (D-4.4 Abierta); descomposicion como mejora (D-4.5)

## Entregables

- **E3** · figura `results/figures/07_incert__varianza_buen_vs_mal_pagador.png` (distribucion
  de la varianza buen vs mal pagador).
- **Cruce D-4.3** · figura `results/figures/07_incert__varianza_vs_n_ext_missing.png`
  (varianza <-> `N_EXT_MISSING` / flags `*_missing`).
- **Tabla** · `results/tables/07_incert__incertidumbre_test.csv` con, por fila de test,
  **clase + `p_bar` + `Var[p]` + `N_EXT_MISSING`**.
- **E4** · curva de loss del **2o modelo del error** (unico entrenamiento final del NB 07) ->
  `results/figures/07_incert__curva_loss.png`. **Transversal** segun convenciones (e): el 07
  tambien entrena un modelo final, asi que aporta su curva de loss (el modelo principal se
  **hereda** del NB 06, no se reentrena aqui).
- **Reflexion** (presentacion) · incertidumbre <-> `EXT_SOURCE` (¿mas duda donde faltan fuentes?).

**Dependencias.** Aguas arriba: **06** (hereda topologia + dropout, cruce **D-3.2 <-> D-4.1**)
y **03** (baseline de referencia). **Sin tareas aguas abajo.**